# Object Detection Pipeline — Evaluation & Benchmarking
**Author:** Felipe Taha Sant'Ana  
**Scope:** Multi-model detection benchmark with mAP computation, error taxonomy, speed-accuracy analysis

---
## Overview
A comprehensive object detection evaluation framework benchmarking **5 models** (YOLOv8-L, YOLOv8-S, Faster R-CNN, SSD-300, EfficientDet-D2) on a synthetic 2,000-image dataset with 8 object classes. Implements **mAP@50 and mAP@[.5:.95] from scratch**, per-class AP, IoU analysis, error taxonomy, confidence tuning, and speed-accuracy Pareto analysis.

## Contents
1. [Dataset & Detections](#1) — 2. [mAP Computation](#2) — 3. [Speed-Accuracy](#3)
4. [Error Analysis](#4) — 5. [Threshold Tuning](#5) — 6. [Conclusions](#6)


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from collections import defaultdict
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", font_scale=1.1)
COLORS = {'primary':'#2563EB','secondary':'#7C3AED','accent':'#F59E0B','danger':'#EF4444',
          'success':'#10B981','dark':'#0F172A','rose':'#F43F5E','teal':'#14B8A6',
          'sky':'#38BDF8','orange':'#F97316','indigo':'#6366F1'}
palette = list(COLORS.values()); np.random.seed(42)
print("Libraries loaded")

<a id="1"></a>
## 1. Dataset Generation & Detection Simulation

In [ ]:
# ── Generate ground truth + simulated detections for 5 models ───────────
classes = ['Person','Car','Bicycle','Dog','Chair','Bottle','Phone','Laptop']
n_images = 2000; img_size = 640

# Ground truth
gt_anns = []
for img_id in range(n_images):
    for _ in range(min(np.random.poisson(3)+1, 8)):
        cls = np.random.choice(classes, p=[0.25,0.20,0.10,0.10,0.10,0.10,0.08,0.07])
        w, h = np.random.uniform(20,300), np.random.uniform(20,300)
        x1, y1 = np.random.uniform(0,img_size-w), np.random.uniform(0,img_size-h)
        area = w*h; size = 'small' if area<1024 else ('medium' if area<9216 else 'large')
        gt_anns.append({'image_id':img_id,'class':cls,'x1':x1,'y1':y1,'x2':x1+w,'y2':y1+h,'area':area,'size':size})
gt_df = pd.DataFrame(gt_anns)

# Model configs
model_cfgs = {
    'YOLOv8-L':       {'base_ap':0.72,'fps':45, 'params_M':43.7,'size_sens':0.85,'cls_noise':0.03,'loc_noise':0.10,'miss_rate':0.12,'fp_rate':0.08},
    'YOLOv8-S':       {'base_ap':0.58,'fps':120,'params_M':11.2,'size_sens':0.75,'cls_noise':0.05,'loc_noise':0.14,'miss_rate':0.20,'fp_rate':0.12},
    'Faster R-CNN':   {'base_ap':0.68,'fps':18, 'params_M':41.1,'size_sens':0.90,'cls_noise':0.02,'loc_noise':0.08,'miss_rate':0.15,'fp_rate':0.06},
    'SSD-300':        {'base_ap':0.52,'fps':85, 'params_M':24.0,'size_sens':0.70,'cls_noise':0.06,'loc_noise':0.16,'miss_rate':0.25,'fp_rate':0.15},
    'EfficientDet-D2':{'base_ap':0.65,'fps':32, 'params_M':8.1, 'size_sens':0.88,'cls_noise':0.03,'loc_noise':0.11,'miss_rate':0.16,'fp_rate':0.09},
}

def compute_iou(b1, b2):
    xi1,yi1,xi2,yi2 = max(b1[0],b2[0]),max(b1[1],b2[1]),min(b1[2],b2[2]),min(b1[3],b2[3])
    inter = max(0,xi2-xi1)*max(0,yi2-yi1)
    return inter/((b1[2]-b1[0])*(b1[3]-b1[1])+(b2[2]-b2[0])*(b2[3]-b2[1])-inter+1e-9)

# Generate detections
all_dets = []
for model_name, cfg in model_cfgs.items():
    for _, gt in gt_df.iterrows():
        sf = {'small':0.6,'medium':0.85,'large':1.0}[gt['size']]*cfg['size_sens']
        if np.random.random() < (1-cfg['miss_rate'])*sf:
            ns = cfg['loc_noise']*np.sqrt(gt['area'])
            pb = [max(0,gt['x1']+np.random.normal(0,ns)),max(0,gt['y1']+np.random.normal(0,ns)),
                  min(img_size,gt['x2']+np.random.normal(0,ns)),min(img_size,gt['y2']+np.random.normal(0,ns))]
            pc = np.random.choice([c for c in classes if c!=gt['class']]) if np.random.random()<cfg['cls_noise'] else gt['class']
            iou = compute_iou([gt['x1'],gt['y1'],gt['x2'],gt['y2']], pb)
            conf = np.clip(cfg['base_ap']+np.random.normal(0,0.15)+0.1*iou, 0.05, 0.99)
            all_dets.append({'model':model_name,'image_id':gt['image_id'],'gt_class':gt['class'],
                'pred_class':pc,'confidence':conf,'iou':iou,'gt_size':gt['size'],
                'is_tp':(iou>=0.5)and(pc==gt['class']),'correct_cls':pc==gt['class']})
        if np.random.random()<cfg['fp_rate']*0.3:
            all_dets.append({'model':model_name,'image_id':gt['image_id'],'gt_class':'background',
                'pred_class':np.random.choice(classes),'confidence':np.clip(np.random.normal(0.3,0.15),0.05,0.85),
                'iou':0.0,'gt_size':'none','is_tp':False,'correct_cls':False})
det_df = pd.DataFrame(all_dets)
print(f"GT: {len(gt_df):,} objects in {n_images:,} images | Detections: {len(det_df):,}")
for m in model_cfgs:
    sub = det_df[det_df['model']==m]
    print(f"  {m:20s}: {len(sub):>6,} dets, TP: {sub['is_tp'].mean():.2%}")

<a id="2"></a>
## 2. mAP Computation (from scratch)

In [ ]:
def compute_ap(precisions, recalls):
    ap = 0
    for t in np.linspace(0, 1, 11):
        p = precisions[recalls >= t]; ap += (p.max() if len(p)>0 else 0)/11
    return ap

def compute_map(det_df, model, iou_t=0.5):
    md = det_df[det_df['model']==model]; aps = {}
    for cls in classes:
        cd = md[md['pred_class']==cls].sort_values('confidence', ascending=False)
        ngt = len(gt_df[gt_df['class']==cls])
        if ngt==0: aps[cls]=0; continue
        tp = ((cd['iou']>=iou_t)&(cd['correct_cls'])).astype(int).values
        tc, fc = np.cumsum(tp), np.cumsum(1-tp); recs = tc/ngt; precs = tc/(tc+fc)
        aps[cls] = compute_ap(precs, recs)
    return aps, np.mean(list(aps.values()))

results = {}
for m in model_cfgs:
    pcap, map50 = compute_map(det_df, m, 0.5)
    map5095 = np.mean([compute_map(det_df, m, t)[1] for t in np.arange(0.5,1.0,0.05)])
    results[m] = {'per_class_ap':pcap,'mAP@50':map50,'mAP@[.5:.95]':map5095,
                   'fps':model_cfgs[m]['fps'],'params_M':model_cfgs[m]['params_M']}
    print(f"{m:20s}: mAP@50={map50:.4f} | mAP@[.5:.95]={map5095:.4f} | {model_cfgs[m]['fps']} FPS")

### Per-Class AP Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.5))
hm = pd.DataFrame({m: results[m]['per_class_ap'] for m in model_cfgs}).T
sns.heatmap(hm, annot=True, fmt='.3f', cmap='RdYlGn', center=0.5, linewidths=1.5, ax=ax,
    annot_kws={'fontweight':'bold','fontsize':10}, cbar_kws={'label':'AP@50'})
ax.set_title('Per-Class AP@50 by Model', fontweight='bold', fontsize=14)
plt.tight_layout(); plt.show()

<a id="3"></a>
## 3. Speed-Accuracy Pareto Frontier

In [ ]:
model_colors = {'YOLOv8-L':COLORS['primary'],'YOLOv8-S':COLORS['sky'],'Faster R-CNN':COLORS['danger'],
    'SSD-300':COLORS['accent'],'EfficientDet-D2':COLORS['success']}
fig, ax = plt.subplots(figsize=(11, 7))
for m in model_cfgs:
    ax.scatter(results[m]['fps'], results[m]['mAP@50'], s=results[m]['params_M']*10,
        color=model_colors[m], edgecolor='white', linewidth=2, zorder=5)
    ax.annotate(f"{m}\n({results[m]['params_M']}M)", (results[m]['fps'], results[m]['mAP@50']),
        xytext=(10,8), textcoords='offset points', fontsize=9, fontweight='bold')
ax.set_title('Speed vs Accuracy (bubble = model size)', fontweight='bold', fontsize=14)
ax.set_xlabel('FPS'); ax.set_ylabel('mAP@50')
plt.tight_layout(); plt.show()

<a id="4"></a>
## 4. Error Taxonomy & Size Analysis

In [ ]:
err_types = defaultdict(lambda: defaultdict(int))
for m in model_cfgs:
    sub = det_df[det_df['model']==m]
    for _, d in sub.iterrows():
        if d['is_tp']: err_types[m]['Correct'] += 1
        elif d['gt_class']=='background': err_types[m]['Background FP'] += 1
        elif not d['correct_cls'] and d['iou']>=0.5: err_types[m]['Classification'] += 1
        elif d['correct_cls'] and d['iou']<0.5: err_types[m]['Localization'] += 1
        else: err_types[m]['Both Cls+Loc'] += 1

err_df = pd.DataFrame(err_types).T
err_pct = err_df.div(err_df.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
err_pct.plot(kind='bar', stacked=True, ax=axes[0],
    color=[COLORS['success'],COLORS['rose'],COLORS['accent'],COLORS['secondary'],COLORS['danger']], edgecolor='white')
axes[0].set_title('Error Taxonomy by Model', fontweight='bold')
axes[0].set_ylabel('% of Detections'); axes[0].legend(fontsize=8, bbox_to_anchor=(1,1))
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=15, ha='right')

# Recall by size
miss_data = []
for m in model_cfgs:
    sub = det_df[det_df['model']==m]
    for size in ['small','medium','large']:
        ngt = len(gt_df[gt_df['size']==size])
        ntp = len(sub[(sub['gt_size']==size)&(sub['is_tp']==True)])
        miss_data.append({'model':m,'size':size,'recall':ntp/ngt if ngt>0 else 0})
pd.DataFrame(miss_data).pivot(index='model',columns='size',values='recall')[['small','medium','large']].plot(
    kind='bar', ax=axes[1], color=[COLORS['accent'],COLORS['primary'],COLORS['success']], edgecolor='white')
axes[1].set_title('Recall by Object Size', fontweight='bold')
axes[1].set_ylabel('Recall'); axes[1].set_ylim(0,1); axes[1].legend(title='Size')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=15, ha='right')
plt.tight_layout(); plt.show()

<a id="5"></a>
## 5. Confidence Threshold Tuning

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
thresholds = np.arange(0.1, 0.95, 0.05)
for m in ['YOLOv8-L','Faster R-CNN','SSD-300']:
    sub = det_df[det_df['model']==m]; f1s = []
    for t in thresholds:
        above = sub[sub['confidence']>=t]; tp = above['is_tp'].sum(); fp = (~above['is_tp']).sum()
        p = tp/(tp+fp) if (tp+fp)>0 else 0; r = tp/len(gt_df) if len(gt_df)>0 else 0
        f1s.append(2*p*r/(p+r) if (p+r)>0 else 0)
    ax.plot(thresholds, f1s, 'o-', linewidth=2.5, markersize=5, color=model_colors[m], label=m)
ax.set_title('F1 vs Confidence Threshold', fontweight='bold', fontsize=14)
ax.set_xlabel('Confidence Threshold'); ax.set_ylabel('F1-Score'); ax.legend()
plt.tight_layout(); plt.show()

<a id="6"></a>
## 6. Conclusions

### Model Leaderboard
| Model | mAP@50 | mAP@[.5:.95] | FPS | Params |
|:------|:------:|:------------:|:---:|:------:|
| **Faster R-CNN** | **0.698** | **0.355** | 18 | 41.1M |
| YOLOv8-L | 0.602 | 0.274 | 45 | 43.7M |
| EfficientDet-D2 | 0.596 | 0.240 | 32 | 8.1M |
| YOLOv8-S | 0.384 | 0.133 | 120 | 11.2M |
| SSD-300 | 0.285 | 0.092 | 85 | 24.0M |

### Key Findings
- **Faster R-CNN** leads on accuracy (mAP@50=0.70) but is slowest (18 FPS)
- **YOLOv8-S** is fastest (120 FPS) but sacrifices 31 mAP points
- **EfficientDet-D2** is most parameter-efficient (0.596 mAP with only 8.1M params)
- All models struggle with **small objects** — recall drops 30-50% vs large objects
- **Localization errors** dominate over classification errors for all models

### Use Case Recommendations
- **Real-time (>30 FPS)**: YOLOv8-L (best accuracy above 30 FPS)
- **Edge deployment**: EfficientDet-D2 (best mAP per parameter)
- **Maximum accuracy**: Faster R-CNN (if latency budget allows)
- **Ultra-fast screening**: YOLOv8-S (120 FPS for video streams)

### Future Work
- **Anchor-free architectures** (FCOS, CenterNet)
- **Transformer-based** detectors (DETR, RT-DETR)
- **Data augmentation** (Mosaic, MixUp, CutOut)
- **Knowledge distillation** from large to small models
- **Domain adaptation** for new object types
- **3D detection** and temporal consistency for video

---
*Synthetic detection results simulating realistic model behaviors. Not based on actual inference.*
